In [53]:
import pandas as pd
import numpy as np

import seaborn as sns
import matplotlib.pyplot as plt

import plotly.express as px
import plotly.graph_objects as go
import plotly.figure_factory as ff

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler   # u otros scalers
from sklearn.linear_model import LinearRegression, Lasso, Ridge, ElasticNet, LassoCV, RidgeCV, ElasticNetCV
from sklearn.metrics import mean_squared_error, r2_score
import haversine

In [54]:
### carga datos de dataset en dataframe
file_path= 'uber_fares.csv'

df = pd.read_csv(file_path)

In [55]:
### visualizacion de algunos datos
df.head()

,key,date,fare_amount,pickup_datetime,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,passenger_count
0,24238194,2015-05-07 19:52:06.0000003,7.5,2015-05-07 19:52:06 UTC,-73.999817,40.738354,-73.999512,40.723217,1
1,27835199,2009-07-17 20:04:56.0000002,7.7,2009-07-17 20:04:56 UTC,-73.994355,40.728225,-73.994710,40.750325,1
2,44984355,2009-08-24 21:45:00.00000061,12.9,2009-08-24 21:45:00 UTC,-74.005043,40.740770,-73.962565,40.772647,1
3,25894730,2009-06-26 08:22:21.0000001,5.3,2009-06-26 08:22:21 UTC,-73.976124,40.790844,-73.965316,40.803349,3
4,17610152,2014-08-28 17:47:00.000000188,16.0,2014-08-28 17:47:00 UTC,-73.925023,40.744085,-73.973082,40.761247,5


#### Contexto  
El proyecto trata sobre **Uber Inc.**, la compañía de taxis más grande del mundo. En este trabajo, nuestro objetivo es **predecir la tarifa de futuros viajes**.  

Uber brinda servicio a millones de clientes cada día, por lo que gestionar adecuadamente sus datos es clave para desarrollar nuevas estrategias de negocio y obtener mejores resultados.  

### Variables del conjunto de datos  

**Variables explicativas:**  
- **key**: identificador único de cada viaje.  
- **pickup_datetime**: fecha y hora en que se inició el viaje.  
- **passenger_count**: cantidad de pasajeros en el vehículo (dato ingresado por el conductor).  
- **pickup_longitude**: longitud del punto de inicio del viaje.  
- **pickup_latitude**: latitud del punto de inicio del viaje.  
- **dropoff_longitude**: longitud del punto de destino.  
- **dropoff_latitude**: latitud del punto de destino.  

**Variable objetivo (target):**  
- **fare_amount**: costo del viaje en dólares.  

In [56]:
### Columnas, ¿cuáles son variables numéricas y cuales variables categóricas?
df.columns

Index(['key', 'date', 'fare_amount', 'pickup_datetime', 'pickup_longitude',
       'pickup_latitude', 'dropoff_longitude', 'dropoff_latitude',
       'passenger_count'],
      dtype='object')

In [57]:
df.describe(include='all')

,key,date,fare_amount,pickup_datetime,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,passenger_count
count,2.000000e+05,200000,200000.000000,200000,200000.000000,200000.000000,199999.000000,199999.000000,200000.000000
unique,NaN,200000,NaN,196629,NaN,NaN,NaN,NaN,NaN
top,NaN,2015-05-07 19:52:06.0000003,NaN,2009-02-12 12:46:00 UTC,NaN,NaN,NaN,NaN,NaN
freq,NaN,1,NaN,4,NaN,NaN,NaN,NaN,NaN
mean,2.771250e+07,NaN,11.359955,NaN,-72.527638,39.935885,-72.525292,39.923890,1.684535
std,1.601382e+07,NaN,9.901776,NaN,11.437787,7.720539,13.117408,6.794829,1.385997
min,1.000000e+00,NaN,-52.000000,NaN,-1340.648410,-74.015515,-3356.666300,-881.985513,0.000000
25%,1.382535e+07,NaN,6.000000,NaN,-73.992065,40.734796,-73.991407,40.733823,1.000000
50%,2.774550e+07,NaN,8.500000,NaN,-73.981823,40.752592,-73.980093,40.753042,1.000000
75%,4.155530e+07,NaN,12.500000,NaN,-73.967154,40.767158,-73.963658,40.768001,2.000000


In [58]:
print(sum(df['pickup_longitude'] > 180)) # valores de longitud y latitud por fuera de (-180 - 180) y (-90 - 90), respectivamente
print(sum(df['pickup_latitude'] < -90))
print(sum(df['pickup_longitude'] < -180))
print(sum(df['pickup_latitude'] > 90))
df = df[df['pickup_latitude'].between(-90, 90)] # borrado de los valores por fuera de estos valores, representan un % insignificante
df = df[df['pickup_longitude'].between(-180, 180)]
df = df[df['dropoff_latitude'].between(-90, 90)]
df = df[df['dropoff_longitude'].between(-180, 180)]



0
0
7
4


In [59]:
df.columns[df.isna().any()] 

Index([], dtype='object')

In [60]:
(df.isna().sum() / len(df)) * 100

key                  0.0
date                 0.0
fare_amount          0.0
pickup_datetime      0.0
pickup_longitude     0.0
pickup_latitude      0.0
dropoff_longitude    0.0
dropoff_latitude     0.0
passenger_count      0.0
dtype: float64

In [61]:
df_limpio = df.dropna()



In [62]:
df_limpio.info()

<class 'pandas.core.frame.DataFrame'>
Index: 199987 entries, 0 to 199999
Data columns (total 9 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   key                199987 non-null  int64  
 1   date               199987 non-null  object 
 2   fare_amount        199987 non-null  float64
 3   pickup_datetime    199987 non-null  object 
 4   pickup_longitude   199987 non-null  float64
 5   pickup_latitude    199987 non-null  float64
 6   dropoff_longitude  199987 non-null  float64
 7   dropoff_latitude   199987 non-null  float64
 8   passenger_count    199987 non-null  int64  
dtypes: float64(5), int64(2), object(2)
memory usage: 15.3+ MB


In [63]:
X_train, X_test, y_train, y_test = train_test_split(df_limpio.drop(columns='fare_amount'), df_limpio['fare_amount'], test_size=0.2, random_state=42)

In [64]:
X_train.isna().sum()

key                  0
date                 0
pickup_datetime      0
pickup_longitude     0
pickup_latitude      0
dropoff_longitude    0
dropoff_latitude     0
passenger_count      0
dtype: int64

In [65]:
X_train.describe(include='all')

,key,date,pickup_datetime,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,passenger_count
count,1.599890e+05,159989,159989,159989.000000,159989.000000,159989.000000,159989.000000,159989.000000
unique,NaN,159989,157840,NaN,NaN,NaN,NaN,NaN
top,NaN,2011-11-30 22:10:09.0000002,2014-04-13 18:19:00 UTC,NaN,NaN,NaN,NaN,NaN
freq,NaN,1,4,NaN,NaN,NaN,NaN,NaN
mean,2.772274e+07,NaN,NaN,-72.504047,39.920486,-72.505007,39.919747,1.681659
std,1.601500e+07,NaN,NaN,10.438777,6.110928,10.430695,6.113690,1.305302
min,1.000000e+00,NaN,NaN,-93.824668,-74.015515,-75.458979,-74.015750,0.000000
25%,1.382620e+07,NaN,NaN,-73.992082,40.734786,-73.991375,40.733760,1.000000
50%,2.775593e+07,NaN,NaN,-73.981823,40.752568,-73.980092,40.753036,1.000000
75%,4.156216e+07,NaN,NaN,-73.967174,40.767145,-73.963623,40.768005,2.000000


In [66]:
X_train['day_week_num'] = pd.to_datetime(X_train['date'], utc=True)
X_train['day_week_num'] = X_train['day_week_num'].dt.dayofweek
X_train['day_week_num']

191312    2
90018     1
15045     4
11426     3
136245    6
         ..
119888    6
103702    5
131941    2
146878    2
121967    6
Name: day_week_num, Length: 159989, dtype: int32

In [67]:
X_train['fecha_hora'] = pd.to_datetime(X_train['date'].str.replace(' UTC', ''), utc=True)

X_train['hora'] = X_train['fecha_hora'].dt.hour

etiquetas = ['Madrugada', 'Mañana', 'Tarde', 'Noche']#O LO HACEMOS NUMERICO PARA DESPUÉS PODER HACER CALCULOS DE COSTO + DÍA DE LA SEMANA EN LA FUNCIÓN DE REGRESIÓN?

bins = [0, 6, 12, 18, 24]

X_train['period_of_the_day'] = pd.cut(X_train['hora'], bins=bins, labels=etiquetas, right=False, include_lowest=True)

X_train['period_of_the_day']

191312        Noche
90018        Mañana
15045         Tarde
11426         Noche
136245    Madrugada
            ...    
119888        Noche
103702        Tarde
131941       Mañana
146878        Noche
121967    Madrugada
Name: period_of_the_day, Length: 159989, dtype: category
Categories (4, object): ['Madrugada' < 'Mañana' < 'Tarde' < 'Noche']

In [ ]:
# Cálculo de distancia recorrida en km según latitud y longitud(inicio y destino)
X_train['km_distance'] = X_train.apply(lambda row: haversine.haversine((row['pickup_latitude'], row['pickup_longitude']),(row['dropoff_latitude'], row['dropoff_longitude'])), axis=1)
X_train['km_distance']

191312     3.345969
90018      1.157905
15045     10.128021
11426      1.406828
136245     2.313561
            ...    
119888     6.950110
103702     2.259134
131941     0.805166
146878     0.907206
121967    11.168209
Name: km_distance, Length: 159989, dtype: float64